# SGIP Method Walkthrough (Pipeline Only)
This notebook walks through the exact algorithm in `method_v2.md` using the current codebase. Each step has a markdown explanation, executable code, and visualization.

## Step 0 — Imports and Helper Functions

In [ ]:
import sys
from pathlib import Path

ROOT = Path('.').resolve().parents[0]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch

from configs.pipeline_config import load_pipeline_config
from datasets.dataset import load_dataset
from image_processings.image_pre_seg import image_i_segment
from image_processings.image_pre_seg import change_image_type
from image_processings.info import Info
from debug_tests.run_tta import run_segmentation_with_info
from metrics.visualize import show_original_and_slic, show_slic_and_points, show_combined_plots
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

plt.rcParams['figure.dpi'] = 120

MAIN_DIR = Path('.').resolve()

# --- Visualization helpers ---
def _to_uint8(img):
    if hasattr(img, 'detach'):
        img = img.detach().cpu().numpy()
    img = np.asarray(img)
    if img.ndim == 3 and img.shape[0] == 3 and img.shape[-1] != 3:
        img = np.transpose(img, (1, 2, 0))
    if img.dtype == np.uint8:
        return img
    if img.max() <= 1.0:
        return (img * 255).astype(np.uint8)
    return img.astype(np.uint8)
def _overlay(image, mask, color=(0, 255, 0), alpha=0.4):
    base = _to_uint8(image)
    if base.ndim == 2:
        base = np.repeat(base[..., None], 3, axis=2)
    overlay = base.copy()
    overlay[mask] = color
    return (base * (1 - alpha) + overlay * alpha).astype(np.uint8)

def _draw_points(ax, points, labels, new_points=None):
    if points is None or len(points) == 0:
        return
    pts = np.asarray(points)
    lbl = np.asarray(labels)
    new_points = np.asarray(new_points) if new_points is not None and len(new_points) else np.empty((0, 2))
    pos = pts[lbl == 1]
    neg = pts[lbl == 0]
    # existing positives
    if len(pos):
        ax.scatter(pos[:, 0], pos[:, 1], c='lime', s=20, edgecolors='k', linewidths=0.3)
    # new positives in a different color
    if len(new_points):
        ax.scatter(new_points[:, 0], new_points[:, 1], c='cyan', s=30, edgecolors='k', linewidths=0.4)
    if len(neg):
        ax.scatter(neg[:, 0], neg[:, 1], c='red', s=20, edgecolors='k', linewidths=0.3)

def _show_heatmap(ax, heat, title):
    ax.imshow(heat, cmap='magma')
    ax.set_title(title)
    ax.axis('off')


## Step 1 — Load Config, Model, and One Sample

In [ ]:
# Load pipeline config
pipeline_cfg = load_pipeline_config(ROOT / 'configs' / 'pipeline.json')

# Load constants for model paths
constants = json.loads((ROOT / 'CONSTANT.json').read_text(encoding='utf-8'))

# Build SAM2 predictor
model = build_sam2(constants['model_cfg'], constants['checkpoint'], device='cuda' if torch.cuda.is_available() else 'cpu')
predictor = SAM2ImagePredictor(model)
predictor.model.to('cuda' if torch.cuda.is_available() else 'cpu')

# Load dataset and pick one sample
images, gt_masks, names = load_dataset(
    pipeline_cfg.dataset.name,
    target_long_edge=pipeline_cfg.dataset.target_long_edge,
    return_paths=True,
)

idx = 0
image = images[idx]
gt_mask = gt_masks[idx]
name = names[idx]

plt.figure(figsize=(4, 4))
plt.imshow(_to_uint8(image))
plt.title(f'Input image: {name}')
plt.axis('off')
plt.show()


## Step 2 — Preprocess (Resize + SLIC)

In [ ]:
pre_segment = image_i_segment(
    image=image,
    new_size_of_image=pipeline_cfg.preprocessing.image_size,
    num_node_for_graph=pipeline_cfg.preprocessing.num_graph_nodes,
    compactness_in_SLIC=pipeline_cfg.preprocessing.slic.compactness,
    sigma_in_SLIC=pipeline_cfg.preprocessing.slic.sigma,
    min_size_factor_in_SLIC=pipeline_cfg.preprocessing.slic.min_size_factor,
    max_size_factor_in_SLIC=pipeline_cfg.preprocessing.slic.max_size_factor,
)

# Use the same conversion as the pipeline
img_resized = change_image_type(pre_segment.image_resized, 'np.array')
seg_tensor = pre_segment.segment_without_padding
segments = seg_tensor.detach().cpu().numpy() if hasattr(seg_tensor, 'detach') else np.asarray(seg_tensor)

# Visualize original + SLIC boundaries
show_original_and_slic(_to_uint8(img_resized), segments, need_show=True)


## Step 3 — Initialize Prompts

In [ ]:
info = Info(
    segment=segments,
    logits=None,
    image=_to_uint8(img_resized),
    graph=pre_segment.graph,
    settings=pipeline_cfg.algorithm,
    debug_mode=False,
    mask_prompt_source=pipeline_cfg.sam.mask_prompt_source,
)

initial_bundle = info.build_initial_prompts()

# Visualize SLIC + prompt points (initial points)
init_pos = [r['point'] for r in info.positive_point_records if r['iteration'] == 0]
show_slic_and_points(_to_uint8(img_resized), segments, initial_bundle.points, initial_bundle.labels)


## Step 4 — Run Full Pipeline (to collect history)

In [ ]:
base_mask, history, vis_image, segments, info = run_segmentation_with_info(
    image, pipeline_cfg, predictor
)
print(f'Total steps: {len(history)}')


## Step 4.5 — Candidate Pool Debug (first 2 iterations)

In [ ]:
# Candidate pool visualization (first 2 iterations)
info_dbg = Info(
    segment=segments,
    logits=None,
    image=_to_uint8(img_resized),
    graph=pre_segment.graph,
    settings=pipeline_cfg.algorithm,
    debug_mode=False,
    mask_prompt_source=pipeline_cfg.sam.mask_prompt_source,
)

max_debug_steps = min(2, len(history))

for step_idx in range(1, max_debug_steps):
    info_dbg.update_from_logits(history[step_idx - 1].logits)
    candidates = info_dbg.get_candidates()
    print(f"Step {step_idx}: candidate_top_k={pipeline_cfg.algorithm.candidate_top_k}, candidates={len(candidates)}")

    # visualize each candidate in the pool (up to top 3)
    show_n = min(3, len(candidates))
    fig, axes = plt.subplots(show_n, 3, figsize=(12, 4 * show_n))
    if show_n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, cand in enumerate(candidates[:show_n]):
        bundle = info_dbg.build_prompts(candidate_id=cand.node_id)
        logits, scores, _ = predictor.predict(
            point_coords=bundle.points,
            point_labels=bundle.labels,
            box=None,
            mask_input=None,
            multimask_output=pipeline_cfg.sam.multimask_output,
            return_logits=True,
        )
        score_val = scores[0] if hasattr(scores, '__len__') else scores
        score_arr = np.asarray(score_val).reshape(-1)
        score = float(score_arr[0]) if score_arr.size else 0.0
        mask = logits[0] > 0

        # column 0: prompts on image
        axes[row, 0].imshow(_to_uint8(vis_image))
        _draw_points(axes[row, 0], bundle.points, bundle.labels)
        axes[row, 0].set_title(f"cand {cand.node_id} prompts")
        axes[row, 0].axis('off')

        # column 1: logits heatmap
        _show_heatmap(axes[row, 1], logits[0], f"cand {cand.node_id} logits")

        # column 2: mask overlay + score
        axes[row, 2].imshow(_overlay(vis_image, mask))
        axes[row, 2].set_title(f"mask (score={score:.4f})")
        axes[row, 2].axis('off')

        print(f"  cand_id={cand.node_id} score={score:.4f}")

    plt.tight_layout()
    plt.show()

    # commit the selected candidate from actual history to keep alignment
    if history[step_idx].candidate_id is not None:
        info_dbg.commit_candidate(history[step_idx].candidate_id)


## Step 5 — Visualize Each Iteration (Prompts / Logits / Mask / Prompt Mask)

In [ ]:
max_steps = None  # set a number to limit
steps = history if max_steps is None else history[:max_steps]

num_steps = len(steps)
fig, axes = plt.subplots(num_steps, 4, figsize=(16, 4 * num_steps))
if num_steps == 1:
    axes = np.expand_dims(axes, axis=0)

for row, step in enumerate(steps):
    # new positive points in this iteration (from Info records)
    new_pos = [r['point'] for r in info.positive_point_records if r['iteration'] == row]

    # Image + prompts (new points in cyan)
    axes[row, 0].imshow(_to_uint8(vis_image))
    _draw_points(axes[row, 0], step.prompts.points, step.prompts.labels, new_points=new_pos)
    axes[row, 0].set_title(f'step {row}: prompts')
    axes[row, 0].axis('off')

    # Logits heatmap
    _show_heatmap(axes[row, 1], step.logits, f'step {row}: logits')

    # Mask prompt (if any)
    mask_prompt = step.prompts.mask_prompt
    if mask_prompt is not None:
        _show_heatmap(axes[row, 2], mask_prompt, f'step {row}: mask_prompt')
    else:
        axes[row, 2].text(0.5, 0.5, 'None', ha='center', va='center')
        axes[row, 2].set_title(f'step {row}: mask_prompt')
        axes[row, 2].axis('off')

    # Mask overlay
    axes[row, 3].imshow(_overlay(vis_image, step.mask))
    axes[row, 3].set_title(f'step {row}: mask')
    axes[row, 3].axis('off')

    # Convex hull trigger notice (based on records)
    if any(r.get('convex_hull_triggered') for r in info.positive_point_records if r['iteration'] == row):
        print(f"Step {row}: convex hull applied to mask_prompt")

plt.tight_layout()
plt.show()


## Step 6 — Final Selection Result

In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(_overlay(vis_image, base_mask))
plt.title('Final selected mask')
plt.axis('off')
plt.show()
